In [ ]:
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
# %%writefile arc_aif.py
# import numpy as np
# from scipy.ndimage import label, find_objects
# from collections import Counter

# # -------------------------------------------------------------------
# # 1. Fast Object Extraction (Using C-optimized scipy)
# # -------------------------------------------------------------------
# def extract_object_features(grid):
#     """Extracts connected components (objects) from a grid.
#     Returns: (num_objects, avg_size, list_of_color_hists)
#     """
#     if grid is None or grid.size == 0:
#         return 0, 0, []
    
#     binary = (grid > 0).astype(np.int32)
#     labeled, num_features = label(binary)
    
#     if num_features == 0:
#         return 0, 0, []
    
#     slices = find_objects(labeled)
#     sizes = []
#     color_hists = []
    
#     for sl in slices:
#         # Extract the subgrid for this object
#         obj_mask = labeled[sl] > 0
#         obj_grid = grid[sl][obj_mask]
        
#         # Size (number of cells)
#         sizes.append(np.sum(obj_mask))
        
#         # Color histogram (count of each color 1-9)
#         hist = np.bincount(obj_grid, minlength=10)[1:]  # exclude 0
#         color_hists.append(hist)
    
#     avg_size = np.mean(sizes) if sizes else 0
#     return num_features, avg_size, color_hists


# def compute_color_mapping(hists_in, hists_out):
#     """Attempts to find a color shift (e.g., 1->4, 2->5) between input and output objects."""
#     shifts = []
#     for h_in, h_out in zip(hists_in, hists_out):
#         in_colors = np.where(h_in > 0)[0] + 1
#         out_colors = np.where(h_out > 0)[0] + 1
#         if len(in_colors) == len(out_colors) and len(in_colors) > 0:
#             sorted_in = sorted(in_colors)
#             sorted_out = sorted(out_colors)
#             diffs = np.array(sorted_out) - np.array(sorted_in)
#             if np.all(diffs == diffs[0]) and len(diffs) > 0:
#                 shifts.append(diffs[0])
#     if shifts:
#         return Counter(shifts).most_common(1)[0][0]
#     return 0


# # -------------------------------------------------------------------
# # 2. Main AIF Analysis Function
# # -------------------------------------------------------------------
# def run_aif_analysis(train_data, test_input):
#     """Main function. Returns dict with: confidence, grid, expected_count, color_shift, is_solved."""
#     try:
#         train_features = []
#         input_counts = []
#         output_counts = []
#         all_avg_sizes = []
#         color_shifts = []

#         # Step 1: Analyze all 4 training pairs
#         for ex in train_data:
#             inp_grid = np.array(ex["input"])
#             out_grid = np.array(ex["output"])
            
#             num_in, avg_in, h_in = extract_object_features(inp_grid)
#             num_out, avg_out, h_out = extract_object_features(out_grid)
            
#             train_features.append({
#                 "in": {"num": num_in, "avg": avg_in, "hists": h_in},
#                 "out": {"num": num_out, "avg": avg_out, "hists": h_out}
#             })
#             input_counts.append(num_in)
#             output_counts.append(num_out)
#             all_avg_sizes.append(avg_in)
            
#             if h_in and h_out and len(h_in) == len(h_out):
#                 shift = compute_color_mapping(h_in, h_out)
#                 if shift != 0:
#                     color_shifts.append(shift)

#         # Step 2: Check structural consistency across ALL 4 examples
#         counts_match = (np.std(input_counts) == 0 and np.std(output_counts) == 0 and 
#                         input_counts[0] == output_counts[0])
#         avg_size_stable = np.std(all_avg_sizes) < 0.5
#         expected_count = int(np.mean(input_counts)) if input_counts else 0
#         color_shift = Counter(color_shifts).most_common(1)[0][0] if color_shifts else 0

#         predicted_grid = None
#         confidence = 0.0

#         # Step 3: If consistent, try to predict via simple translation/copy
#         if counts_match and avg_size_stable and expected_count > 0:
#             # Use first training example to compute object movement deltas
#             in_slices = find_objects(label((np.array(train_data[0]["input"]) > 0).astype(int))[0])
#             out_slices = find_objects(label((np.array(train_data[0]["output"]) > 0).astype(int))[0])
            
#             if len(in_slices) == len(out_slices) == expected_count:
#                 test_grid = np.array(test_input)
#                 predicted_grid = np.zeros_like(test_grid)
                
#                 test_binary = (test_grid > 0).astype(int)
#                 test_labels, test_num = label(test_binary)
                
#                 if test_num == expected_count:
#                     test_slices = find_objects(test_labels)
#                     matches = 0
                    
#                     for i, sl_in_test in enumerate(test_slices):
#                         if i >= len(out_slices):
#                             break
#                         # Get object pixels from test
#                         obj_mask = test_labels[sl_in_test] > 0
#                         obj_colors = test_grid[sl_in_test][obj_mask]
                        
#                         # Compute centroid shift from training example 0
#                         dy = out_slices[i][0].start - in_slices[i][0].start
#                         dx = out_slices[i][1].start - in_slices[i][1].start
                        
#                         # Apply shift to test object position
#                         h = sl_in_test[0].stop - sl_in_test[0].start
#                         w = sl_in_test[1].stop - sl_in_test[1].start
#                         new_y = sl_in_test[0].start + dy
#                         new_x = sl_in_test[1].start + dx
                        
#                         # Clamp and place into predicted grid
#                         if (new_y >= 0 and new_x >= 0 and 
#                             new_y + h <= predicted_grid.shape[0] and 
#                             new_x + w <= predicted_grid.shape[1]):
#                             predicted_grid[new_y:new_y+h, new_x:new_x+w][obj_mask] = obj_colors
#                             matches += 1
                    
#                     if matches == expected_count:
#                         confidence = 1.0
#                     else:
#                         confidence = 0.3
#                         predicted_grid = None

#         # Step 4: Return result
#         if predicted_grid is not None and np.sum(predicted_grid) > 0:
#             return {
#                 "confidence": confidence,
#                 "grid": predicted_grid.tolist(),
#                 "expected_count": expected_count,
#                 "color_shift": color_shift,
#                 "is_solved": confidence > 0.95
#             }
#         else:
#             return {
#                 "confidence": max(0.0, 0.1 * (counts_match + avg_size_stable) / 2),
#                 "grid": None,
#                 "expected_count": expected_count,
#                 "color_shift": color_shift,
#                 "is_solved": False
#             }
            
#     except Exception as e:
#         print(f"AIF Error: {e}")
#         return {
#             "confidence": 0.0,
#             "grid": None,
#             "expected_count": 0,
#             "color_shift": 0,
#             "is_solved": False
#         }

In [ ]:
%%writefile arc_aif.py
import numpy as np
from scipy.ndimage import label, find_objects, center_of_mass
from collections import Counter
import math

# -------------------------------------------------------------------
# 1. Fast Object Extraction (same as before, but returns more)
# -------------------------------------------------------------------
def extract_object_features(grid):
    """Returns: (num_objects, avg_size, list_of_masks, list_of_centroids, color_hists)"""
    if grid is None or grid.size == 0:
        return 0, 0, [], [], []
    
    binary = (grid > 0).astype(np.int32)
    labeled, num_features = label(binary)
    
    if num_features == 0:
        return 0, 0, [], [], []
    
    slices = find_objects(labeled)
    sizes = []
    masks = []
    centroids = []
    color_hists = []
    
    for sl in slices:
        obj_mask = labeled[sl] > 0
        masks.append(obj_mask)
        sizes.append(np.sum(obj_mask))
        # Centroid (row, col)
        cy, cx = center_of_mass(labeled[sl], labels=labeled[sl], index=np.unique(labeled[sl][obj_mask])[0])
        centroids.append((cy, cx))
        obj_grid = grid[sl][obj_mask]
        hist = np.bincount(obj_grid, minlength=10)[1:]
        color_hists.append(hist)
    
    avg_size = np.mean(sizes) if sizes else 0
    return num_features, avg_size, masks, centroids, color_hists


def compute_color_mapping(hists_in, hists_out):
    """Finds color shifts between two sets of object histograms."""
    shifts = []
    for h_in, h_out in zip(hists_in, hists_out):
        in_colors = np.where(h_in > 0)[0] + 1
        out_colors = np.where(h_out > 0)[0] + 1
        if len(in_colors) == len(out_colors) and len(in_colors) > 0:
            sorted_in = sorted(in_colors)
            sorted_out = sorted(out_colors)
            diffs = np.array(sorted_out) - np.array(sorted_in)
            if np.all(diffs == diffs[0]) and len(diffs) > 0:
                shifts.append(diffs[0])
    if shifts:
        return Counter(shifts).most_common(1)[0][0]
    return 0


# -------------------------------------------------------------------
# 2. Rule Inference Engine (The World Model)
# -------------------------------------------------------------------
def bresenham_line(p1, p2, shape):
    """Generate a binary mask of a line between two points."""
    r1, c1 = int(round(p1[0])), int(round(p1[1]))
    r2, c2 = int(round(p2[0])), int(round(p2[1]))
    rr, cc = [], []
    dr = abs(r2 - r1)
    dc = -abs(c2 - c1)
    sr = 1 if r1 < r2 else -1
    sc = 1 if c1 < c2 else -1
    err = dr + dc
    while True:
        rr.append(r1)
        cc.append(c1)
        if r1 == r2 and c1 == c2:
            break
        e2 = 2 * err
        if e2 >= dc:
            err += dc
            r1 += sr
        if e2 <= dr:
            err += dr
            c1 += sc
    mask = np.zeros(shape, dtype=bool)
    valid_r = np.clip(rr, 0, shape[0]-1)
    valid_c = np.clip(cc, 0, shape[1]-1)
    mask[valid_r, valid_c] = True
    return mask


def match_translation(input_masks, input_centroids, output_masks, output_centroids, grid_shape):
    """Rule 1: Objects moved unchanged (translation)."""
    if len(input_masks) != len(output_masks) or len(input_masks) == 0:
        return None, 0.0
    
    shifts = []
    for i in range(len(input_masks)):
        dy = output_centroids[i][0] - input_centroids[i][0]
        dx = output_centroids[i][1] - input_centroids[i][1]
        shifts.append((dy, dx))
    
    # Check if the shift is consistent across all objects
    if all(s == shifts[0] for s in shifts):
        dy, dx = shifts[0]
        return {"type": "TRANSLATE", "dy": dy, "dx": dx}, 1.0
    return None, 0.0


def match_symmetry(input_masks, output_masks, grid_shape):
    """Rule 2: Mirror / Symmetry."""
    if len(input_masks) == 0 or len(output_masks) == 0:
        return None, 0.0
    
    # Build full input mask from its components
    full_in = np.zeros(grid_shape, dtype=bool)
    for m in input_masks:
        # We need the absolute positions, but masks are cropped.
        # For simplicity, we compare the full grids if only one object.
        # For multiple objects, we check if output masks are mirrored versions of input masks.
        pass
    
    # Simplified: check if output is a horizontal or vertical concatenation of input and its flip
    # (Common ARC symmetry task: half shape -> full shape)
    if len(input_masks) == 1:
        # Try to reconstruct output by flipping the input mask
        in_mask_full = np.zeros(grid_shape, dtype=bool)
        # We'll just use the full input grid (train_data[0]["input"]) outside, so this is tricky.
        # Better to handle this at the grid level in the main function.
        return None, 0.0
    return None, 0.0


def match_arithmetic(input_masks, input_centroids, output_masks, output_centroids, grid_shape):
    """Rule 3: Duplication (count multiplied) or Merging (count divided)."""
    n_in = len(input_masks)
    n_out = len(output_masks)
    if n_in == 0 or n_out == 0:
        return None, 0.0
    
    # Duplication: n_out is multiple of n_in
    if n_out % n_in == 0 and n_out > n_in:
        factor = n_out // n_in
        # Check if every input object is replicated 'factor' times in output
        # We check if output objects are identical to input objects (same shape/color)
        # This is hard without full object matching.
        # We'll return a generic "DUPLICATE" if counts are a clean multiple.
        return {"type": "DUPLICATE", "factor": factor}, 0.8
    # Merging: n_in % n_out == 0
    elif n_in % n_out == 0 and n_in > n_out:
        factor = n_in // n_out
        return {"type": "MERGE", "factor": factor}, 0.8
    return None, 0.0


def match_line_drawing(input_centroids, output_masks, grid_shape):
    """Rule 4: Line drawing between points."""
    if len(input_centroids) == 2 and len(output_masks) == 1:
        # Generate line between the two centroids
        line_mask = bresenham_line(input_centroids[0], input_centroids[1], grid_shape)
        # Compare to output mask (we only have mask, need full grid to compare)
        # We'll just check area: line area should roughly match output area
        out_area = np.sum(output_masks[0])
        line_area = np.sum(line_mask)
        if abs(out_area - line_area) / max(1, line_area) < 0.2:
            return {"type": "DRAW_LINE", "p1": input_centroids[0], "p2": input_centroids[1]}, 1.0
    return None, 0.0


def match_logical_op(input_grids, output_grid):
    """Rule 5: Logical operations (XOR, AND, OR, SUB)."""
    if len(input_grids) != 2:
        return None, 0.0
    A = (input_grids[0] > 0).astype(bool)
    B = (input_grids[1] > 0).astype(bool)
    out = (output_grid > 0).astype(bool)
    
    if np.array_equal(out, np.logical_xor(A, B)):
        return {"type": "XOR"}, 1.0
    if np.array_equal(out, np.logical_and(A, B)):
        return {"type": "AND"}, 1.0
    if np.array_equal(out, np.logical_or(A, B)):
        return {"type": "OR"}, 1.0
    if np.array_equal(out, np.logical_and(A, np.logical_not(B))):
        return {"type": "SUBTRACT"}, 1.0
    if np.array_equal(out, np.logical_and(B, np.logical_not(A))):
        return {"type": "SUBTRACT_REV"}, 1.0
    return None, 0.0


def match_tiling(input_masks, output_masks, grid_shape):
    """Rule 6: Tiling / Fractal."""
    if len(input_masks) == 1 and len(output_masks) > 1:
        # Check if output objects are all identical to the input object
        # We can't easily compare shapes without full reconstruction.
        # But if output has many objects and average size matches input, it's likely tiling.
        return {"type": "TILE"}, 0.7
    return None, 0.0


def apply_rule(rule, test_input, train_data):
    """Apply the inferred rule to the test input to generate the output."""
    if rule is None:
        return None
    
    rule_type = rule.get("type")
    
    if rule_type == "TRANSLATE":
        dy, dx = rule["dy"], rule["dx"]
        # Shift the entire test grid (or its objects)
        # Simpler: shift the whole grid
        grid = np.array(test_input)
        shifted = np.zeros_like(grid)
        h, w = grid.shape
        for r in range(h):
            for c in range(w):
                if grid[r, c] > 0:
                    nr, nc = r + int(dy), c + int(dx)
                    if 0 <= nr < h and 0 <= nc < w:
                        shifted[nr, nc] = grid[r, c]
        return shifted
    
    if rule_type == "DRAW_LINE":
        p1, p2 = rule["p1"], rule["p2"]
        line_mask = bresenham_line(p1, p2, np.array(test_input).shape)
        # Apply colors from input objects (or just use color 1)
        output = np.zeros_like(test_input, dtype=int)
        # Use the first input object's color if available
        color = 1
        # If there are objects, use their colors
        grid = np.array(test_input)
        colors = grid[grid > 0]
        if len(colors) > 0:
            color = int(colors[0])
        output[line_mask] = color
        return output
    
    if rule_type == "DUPLICATE":
        factor = rule["factor"]
        grid = np.array(test_input)
        # Simply tile the grid in a pattern (e.g., 2x2 or 1xN)
        # For simplicity, just repeat the whole grid
        h, w = grid.shape
        # Try to arrange in a square-ish grid
        rows = int(np.ceil(np.sqrt(factor)))
        cols = int(np.ceil(factor / rows))
        out_h = h * rows
        out_w = w * cols
        output = np.zeros((out_h, out_w), dtype=int)
        for i in range(rows):
            for j in range(cols):
                if i * cols + j < factor:
                    output[i*h:(i+1)*h, j*w:(j+1)*w] = grid
        return output
    
    if rule_type in ["XOR", "AND", "OR", "SUBTRACT", "SUBTRACT_REV"]:
        # This rule applies to the input itself (2 input grids in training)
        # For test, we only have one input. We can't apply logical ops to single input.
        return None
    
    if rule_type == "TILE":
        grid = np.array(test_input)
        # Tile the input pattern across the whole output canvas
        # We need output size, but we don't know it. Use training output size.
        # Use first training output as template size
        train_out = np.array(train_data[0]["output"])
        h_out, w_out = train_out.shape
        h_in, w_in = grid.shape
        output = np.zeros((h_out, w_out), dtype=int)
        for r in range(0, h_out, h_in):
            for c in range(0, w_out, w_in):
                rh = min(h_in, h_out - r)
                rw = min(w_in, w_out - c)
                output[r:r+rh, c:c+rw] = grid[:rh, :rw]
        return output
    
    return None


# -------------------------------------------------------------------
# 3. Main AIF Analysis Function (Upgraded)
# -------------------------------------------------------------------
def run_aif_analysis(train_data, test_input):
    """Full Active Inference World Model.
    Tries multiple rules, picks the one that perfectly explains all training examples.
    """
    try:
        train_features = []
        input_counts = []
        output_counts = []
        all_avg_sizes = []
        color_shifts = []

        # Step 1: Extract features from all training pairs
        for ex in train_data:
            inp_grid = np.array(ex["input"])
            out_grid = np.array(ex["output"])
            
            num_in, avg_in, masks_in, cents_in, h_in = extract_object_features(inp_grid)
            num_out, avg_out, masks_out, cents_out, h_out = extract_object_features(out_grid)
            
            train_features.append({
                "in": {"num": num_in, "avg": avg_in, "masks": masks_in, "cents": cents_in, "grid": inp_grid, "hists": h_in},
                "out": {"num": num_out, "avg": avg_out, "masks": masks_out, "cents": cents_out, "grid": out_grid, "hists": h_out}
            })
            input_counts.append(num_in)
            output_counts.append(num_out)
            all_avg_sizes.append(avg_in)
            
            if h_in and h_out and len(h_in) == len(h_out):
                shift = compute_color_mapping(h_in, h_out)
                if shift != 0:
                    color_shifts.append(shift)

        # Step 2: Check consistency across all examples
        counts_match = (np.std(input_counts) == 0 and np.std(output_counts) == 0)
        avg_size_stable = np.std(all_avg_sizes) < 0.5
        expected_count = int(np.mean(input_counts)) if input_counts else 0
        color_shift = Counter(color_shifts).most_common(1)[0][0] if color_shifts else 0

        predicted_grid = None
        confidence = 0.0
        best_rule = None

        # Use the first training example to infer the rule
        first_in = train_features[0]["in"]
        first_out = train_features[0]["out"]
        grid_shape = first_in["grid"].shape

        # --- CASCADE OF RULES (BFS over transformations) ---
        
        # Rule 1: Translation
        rule, conf = match_translation(
            first_in["masks"], first_in["cents"],
            first_out["masks"], first_out["cents"],
            grid_shape
        )
        if rule and conf > confidence:
            # Validate on other training examples
            all_match = True
            for feat in train_features[1:]:
                r2, c2 = match_translation(
                    feat["in"]["masks"], feat["in"]["cents"],
                    feat["out"]["masks"], feat["out"]["cents"],
                    grid_shape
                )
                if not r2 or r2["type"] != "TRANSLATE":
                    all_match = False
                    break
            if all_match:
                best_rule = rule
                confidence = conf

        # Rule 2: Arithmetic (Duplication / Merging)
        if not best_rule:
            rule, conf = match_arithmetic(
                first_in["masks"], first_in["cents"],
                first_out["masks"], first_out["cents"],
                grid_shape
            )
            if rule and conf > confidence:
                all_match = True
                for feat in train_features[1:]:
                    r2, c2 = match_arithmetic(
                        feat["in"]["masks"], feat["in"]["cents"],
                        feat["out"]["masks"], feat["out"]["cents"],
                        grid_shape
                    )
                    if not r2 or r2["type"] != rule["type"]:
                        all_match = False
                        break
                if all_match:
                    best_rule = rule
                    confidence = conf

        # Rule 3: Line Drawing
        if not best_rule and first_in["num"] == 2 and first_out["num"] == 1:
            rule, conf = match_line_drawing(
                first_in["cents"],
                first_out["masks"],
                grid_shape
            )
            if rule and conf > confidence:
                all_match = True
                for feat in train_features[1:]:
                    if feat["in"]["num"] != 2 or feat["out"]["num"] != 1:
                        all_match = False
                        break
                    r2, c2 = match_line_drawing(
                        feat["in"]["cents"],
                        feat["out"]["masks"],
                        grid_shape
                    )
                    if not r2:
                        all_match = False
                        break
                if all_match:
                    best_rule = rule
                    confidence = conf

        # Rule 4: Logical Operations (XOR, AND, etc.)
        if not best_rule and len(train_data[0]["input"]) == 2:  # two input grids?
            # Actually ARC train data is a list of dicts, each with "input" and "output".
            # Logical ops usually appear with multiple input grids in one pair? 
            # ARC standard: each pair is one input -> one output. So this doesn't apply.
            pass

        # Rule 5: Tiling
        if not best_rule and first_in["num"] == 1 and first_out["num"] > 1:
            rule, conf = match_tiling(
                first_in["masks"],
                first_out["masks"],
                grid_shape
            )
            if rule and conf > confidence:
                all_match = True
                for feat in train_features[1:]:
                    if feat["in"]["num"] != 1 or feat["out"]["num"] <= 1:
                        all_match = False
                        break
                if all_match:
                    best_rule = rule
                    confidence = conf

        # Step 3: Apply the best rule to the test input
        if best_rule:
            predicted_grid = apply_rule(best_rule, test_input, train_data)
            # If we got a valid grid, set high confidence
            if predicted_grid is not None and np.sum(predicted_grid) > 0:
                confidence = max(confidence, 0.95)

        # Step 4: Fallback to simple object-count pruning if no rule matched
        if predicted_grid is None:
            if counts_match and avg_size_stable and expected_count > 0:
                # Simple copy of test grid (fallback)
                predicted_grid = np.array(test_input)
                confidence = max(0.0, 0.1 * (counts_match + avg_size_stable) / 2)
            else:
                confidence = 0.0

        # Step 5: Return result
        if predicted_grid is not None and np.sum(predicted_grid) > 0:
            return {
                "confidence": confidence,
                "grid": predicted_grid.tolist(),
                "expected_count": expected_count,
                "color_shift": color_shift,
                "rule_type": best_rule["type"] if best_rule else "UNKNOWN",
                "is_solved": confidence > 0.95
            }
        else:
            return {
                "confidence": 0.0,
                "grid": None,
                "expected_count": expected_count,
                "color_shift": color_shift,
                "rule_type": "NONE",
                "is_solved": False
            }
            
    except Exception as e:
        print(f"AIF Error: {e}")
        return {
            "confidence": 0.0,
            "grid": None,
            "expected_count": 0,
            "color_shift": 0,
            "rule_type": "ERROR",
            "is_solved": False
        }

In [ ]:
# %%writefile arc_solver.py
# from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
# from arc_loader import ArcDataset, QwenFormatter

# import gc
# import os
# import io
# import time
# import torch
# import numpy as np
# from tqdm import tqdm
# from datasets import Dataset
# from collections import defaultdict

# from typing import Any, Union
# from transformers import DataCollatorForLanguageModeling

# import logging
# from contextlib import redirect_stdout, redirect_stderr

# from peft import get_peft_model_state_dict, set_peft_model_state_dict

# import bz2
# import pickle

# logging.disable(logging.WARNING)

# ARC_VOCAB = {
#     "0": 0,
#     "1": 1,
#     "2": 2,
#     "3": 3,
#     "4": 4,
#     "5": 5,
#     "6": 6,
#     "7": 7,
#     "8": 8,
#     "9": 9,
#     "Ċ": 10,
#     "<|im_end|>": 15,
# }

# ARC_TOKENS = list(ARC_VOCAB.values())
# USER_TOKEN_ID = 11
# ASSISTANT_TOKEN_ID = 12
# PAD_ID = 13
# EOS_ID = 15


# class UnslothFixedTrainer(UnslothTrainer):

#     # Issue https://github.com/unslothai/unsloth/issues/2435

#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         """Fixed compute_loss that handles Unsloth's view tensor issue"""
#         if self.label_smoother is not None and "labels" in inputs:
#             labels = inputs.pop("labels")
#         else:
#             labels = None
#         outputs = model(**inputs)
#         if labels is not None:
#             unwrapped_model = self.accelerator.unwrap_model(model)
#             if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
#                 loss = self.label_smoother(outputs, labels, shift_labels=True)
#             else:
#                 loss = self.label_smoother(outputs, labels)
#         else:
#             loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
#         # 🔧 KEY FIX: Clone the loss tensor before in-place operations
#         if hasattr(loss, "clone"):
#             loss = loss.clone()  # Converts view tensor to independent tensor
#         # Now safe for DDP gradient scaling
#         if self.accelerator.num_processes > 1:
#             loss = loss * self.accelerator.num_processes
#         return (loss, outputs) if return_outputs else loss


# class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

#     def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
#         batch = super().torch_call(examples)
#         for i in range(len(examples)):
#             labels = batch["input_ids"][i].clone()
#             user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
#             assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
#             start_idx = sorted(user_start_idx + assistant_start_idx)
#             end_idx = np.where(labels == EOS_ID)[0]
#             batch["labels"][i, :] = -100
#             for j, (start, end) in enumerate(zip(start_idx, end_idx)):
#                 assert start < end
#                 if j % 2 == 1:
#                     start += 2
#                     end += 1
#                     batch["labels"][i, start:end] = labels[start:end]
#         return batch


# def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

#     n = logits.size(0)

#     nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

#     suffixes = defaultdict(list)

#     candidates = dict()

#     for i in range(n):
#         candidates[i] = []
#         for t in ARC_TOKENS:
#             score = nll[i, t].item()
#             if score < max_score:
#                 if t == EOS_ID:
#                     suffixes[i].append((score, [t]))
#                 elif max_new_tokens > 1:
#                     candidates[i].append((score, t))

#     for i in range(n):
#         candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
#     while time.time() - start_time < 540 and time.time() < end_time:

#         batch_tokens = []
#         batch_scores = []
#         num_alive_beams = 0

#         for i in range(n):
#             if len(candidates[i]) == 0:
#                 batch_tokens.append(PAD_ID)
#                 batch_scores.append(1000)
#             else:
#                 score, t = candidates[i].pop(0)
#                 batch_tokens.append(t)
#                 batch_scores.append(score)
#                 num_alive_beams += 1

#         if num_alive_beams == 0:
#             break

#         outputs = model(
#             input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
#             position_ids=torch.full((n, 1), pos, device=model.device),
#             past_key_values=cache,
#             return_dict=True,
#             use_cache=True,
#         )

#         next_suffixes = turbo_dfs(
#             model,
#             logits=outputs.logits[:, -1],
#             max_new_tokens=max_new_tokens-1,
#             max_score=max_score,
#             scores=batch_scores,
#             pos=pos+1,
#             cache=outputs.past_key_values,
#             start_time=start_time,
#             end_time=end_time,
#         )

#         for batch_id, beams in next_suffixes.items():
#             for score, suffix_tokens in beams:
#                 suffix_tokens.insert(0, batch_tokens[batch_id])
#                 suffixes[batch_id].append((score, suffix_tokens))

#     return suffixes


# @torch.no_grad()
# def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
#     input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
#     outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
#     suffixes = turbo_dfs(
#         model,
#         logits=outputs.logits[:, -1],
#         max_new_tokens=max_new_tokens,
#         max_score=max_score,
#         scores=[0.0] * input_ids.size(0),
#         pos=input_ids.size(1),
#         cache=outputs.past_key_values,
#         start_time=time.time(),
#         end_time=end_time,
#     )
#     result = []
#     for batch_id, beams in suffixes.items():
#         sorted_beams = sorted(beams, key=lambda x:x[0])
#         result.append((batch_id, sorted_beams))
#     return result


# @torch.no_grad()
# def calc_scores(queries, answers, tokenizer, model):
#     batch_query_tokens = []
#     batch_answer_tokens = []
#     batch_tokens = []
#     batch_lengths = []
#     for query, answer in zip(queries, answers):
#         query_tokens = tokenizer.encode(query)
#         answer_tokens = tokenizer.encode(answer)
#         tokens = query_tokens + answer_tokens
#         batch_query_tokens.append(query_tokens)
#         batch_answer_tokens.append(answer_tokens)
#         batch_tokens.append(tokens)
#         batch_lengths.append(len(tokens))
#     max_len = max(batch_lengths)
#     padded_tokens = []
#     for tokens in batch_tokens:
#         padded = tokens + [PAD_ID] * (max_len - len(tokens))
#         padded_tokens.append(padded)
#     input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
#     outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
#     batch_logits = outputs.logits.float().cpu().log_softmax(-1)
#     result = []
#     for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
#         query_length = len(query_tokens)
#         answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
#         answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
#         result.append(-answer_score.item())
#     return result


# def worker(rank, queue, end_time):

#     rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

#     peft_params = dict(
#         r=256,
#         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
#         lora_alpha=32,
#         lora_dropout=0.0,
#         bias="none",
#         use_gradient_checkpointing=False,
#         random_state=42,
#         use_rslora=True,
#         loftq_config=None,
#     )

#     train_args = dict(
#         per_device_eval_batch_size=1,
#         per_device_train_batch_size=1,
#         gradient_accumulation_steps=1,
#         num_train_epochs=1,
#         warmup_steps=0,
#         warmup_ratio=0.1,
#         max_grad_norm=1.0,
#         learning_rate=5e-5,
#         optim="adamw_torch",
#         weight_decay=0.0,
#         lr_scheduler_type="cosine",
#         seed=42,
#         report_to="none",
#         save_strategy="no",
#         eval_strategy="no",
#         logging_strategy="no",
#         fp16=False,
#         bf16=True,
#         # Disable FSDP (use standard DDP)
#         fsdp="",
#         ddp_find_unused_parameters=False,
#         dataloader_num_workers=0,
#         gradient_checkpointing=False,
#     )

#     max_seq_length = 8192

#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name="/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
#         full_finetuning=False,
#         load_in_4bit=False,
#         local_files_only=True,
#         use_gradient_checkpointing=False,
#         max_seq_length=max_seq_length,
#     )

#     model = FastLanguageModel.get_peft_model(model, **peft_params)

#     for name, param in model.named_parameters():
#         if param.dtype == torch.float32:
#             param.data = param.data.to(torch.bfloat16)

#     default_weights = get_peft_model_state_dict(model, adapter_name="default")
#     default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

#     collator = QwenDataCollatorForCompletionOnlyLM(
#         tokenizer=tokenizer,
#         mlm=False,
#     )

#     formatter = QwenFormatter(tokenizer=tokenizer)

#     max_new_tokens = formatter.max_new_tokens()

#     max_score = -np.log(0.2)

#     if rerun_mode:
#         test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
#     else:
#         test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

#     arc_test_set = ArcDataset.from_file(test_path)

#     dir_outputs = "/kaggle/inference_outputs"
#     os.makedirs(dir_outputs, exist_ok=True)

#     while not queue.empty():

#         if time.time() > end_time:
#             print(f"[Rank {rank}] stop!")
#             break

#         key = queue.get()
#         if key is None:
#             break
        
#         start_time = time.time()
        
#         torch.cuda.reset_peak_memory_stats()

#         load_result = set_peft_model_state_dict(
#             model,
#             default_weights.copy(),
#             adapter_name="default",
#         )

#         model = FastLanguageModel.for_training(model)

#         puzzle_ds = arc_test_set.change_keys([key])

#         train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
#         train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

#         with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
#             trainer = UnslothFixedTrainer(
#                 model=model,
#                 tokenizer=tokenizer,
#                 data_collator=collator,
#                 train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
#                 dataset_text_field="text",
#                 max_seq_length=max_seq_length,
#                 args=UnslothTrainingArguments(**train_args),
#             )

#             stats = trainer.train()

#             model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

#             del trainer

#         model = FastLanguageModel.for_inference(model)
        
#         gc.collect()
#         torch.cuda.empty_cache()
            
#         memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
#         print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

#         torch.cuda.reset_peak_memory_stats()
        
#         print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

#         puzzle_ds_multi = puzzle_ds.split_multi_replies()

#         eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
#         eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

#         test_id_to_subkeys = defaultdict(list)
#         for subkey in sorted(eval_ds.keys):
#             test_id = subkey.split(".")[0].split("_")[1]
#             test_id_to_subkeys[test_id].append(subkey)

#         batches = []
#         for test_id, subkeys in test_id_to_subkeys.items():
#             # 0: permute x 2
#             # 4: rot90.rot90.permute x 2
#             batch = []
#             for offset in [0, 4]:
#                 batch.extend(subkeys[offset:offset+2])
#             batches.append(batch)
#             # 2: permute.rot90 x 2
#             # 6: rot90.rot90.rot90.permute x 2
#             batch = []
#             for offset in [2, 6]:
#                 batch.extend(subkeys[offset:offset+2])
#             batches.append(batch)
#         for test_id, subkeys in test_id_to_subkeys.items():
#             # 8: transpose.permute x 2
#             # 12: transpose.rot90.rot90.permute x 2
#             batch = []
#             for offset in [8, 12]:
#                 batch.extend(subkeys[offset:offset+2])
#             batches.append(batch)
#             # 10: transpose.rot90.permute x 2
#             # 14: transpose.rot90.rot90.rot90.permute x 2
#             batch = []
#             for offset in [10, 14]:
#                 batch.extend(subkeys[offset:offset+2])
#             batches.append(batch)

#         with torch.inference_mode():
                
#             known_scores = {}

#             for subkeys in batches:

#                 spend_time = time.time() - start_time
#                 if spend_time > 1800 or time.time() > end_time:
#                     print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
#                     break

#                 print(f"[Rank {rank}] decoding {subkeys}")

#                 tokens = []
#                 for subkey in subkeys:
#                     data = eval_ds.get(subkey, formatter)
#                     tokens.append(tokenizer.encode(data["input"]))

#                 dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

#                 for subkey_id, scored_beams in dfs_result:

#                     subkey = subkeys[subkey_id]
#                     bk = subkey.split(".")[0]
#                     decoded_result = []

#                     for beam_score, tokens in scored_beams:

#                         array = formatter.convert_tokens_to_array(tokens)
#                         if array is None:
#                             continue

#                         solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

#                         grid_id = (bk, tuple(map(tuple, solution)))

#                         if grid_id in known_scores:
#                             augmented_scores = known_scores[grid_id]
#                         else:
#                             print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
#                             aug_dataset = ArcDataset(
#                                 keys=[bk],
#                                 queries={bk: puzzle_ds_multi.queries.get(bk)},
#                                 replies={bk: [solution.tolist()]},
#                             )
#                             aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
#                             aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
#                             aug_queries = []
#                             aug_answers = []
#                             for augmented_sample in aug_dataset.as_list(formatter):
#                                 aug_queries.append(augmented_sample["input"])
#                                 aug_answers.append(augmented_sample["reply"])
#                             augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
#                             augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
#                             augmented_scores = augmented_scores1 + augmented_scores2
#                             known_scores[grid_id] = augmented_scores
                        
#                         decoded_result.append({
#                             "beam_score": beam_score,
#                             "score_aug": augmented_scores,
#                             "solution": solution,
#                         })

#                     if len(decoded_result):
#                         with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
#                             pickle.dump(decoded_result, f)

#         memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
#         print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
#         spend_time = time.time() - start_time
#         print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter
from arc_aif import run_aif_analysis  # <-- NEW IMPORT

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict
from typing import Any, Union
from transformers import DataCollatorForLanguageModeling
import logging
from contextlib import redirect_stdout, redirect_stderr
from peft import get_peft_model_state_dict, set_peft_model_state_dict
import bz2
import pickle
from concurrent.futures import ThreadPoolExecutor  # <-- NEW
from scipy.ndimage import label  # <-- NEW for pruning

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "Ċ": 10, "<|im_end|>": 15,
}
ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:
    n = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates = dict()
    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))
    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0])
    
    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0
        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1
        if num_alive_beams == 0:
            break
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )
        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )
        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))
    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)
    max_new_tokens = formatter.max_new_tokens()
    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)
    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():
        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])
        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        # ================================================================
        # LAUNCH AIF IN BACKGROUND THREAD
        # ================================================================
        aif_future = None
        executor = ThreadPoolExecutor(max_workers=1)
        try:
            raw_train_data = puzzle_ds.queries[key]['train']
            raw_test_input = puzzle_ds.queries[key]['test'][0]['input']
            aif_future = executor.submit(
                run_aif_analysis,
                raw_train_data,
                raw_test_input
            )
            print(f"[Rank {rank}] AIF background analysis launched for {key}...")
        except Exception as e:
            print(f"[Rank {rank}] Failed to launch AIF: {e}")
            aif_future = None

        # ================================================================
        # TRAINING PHASE (GPU busy for ~3.5 minutes)
        # ================================================================
        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )
            stats = trainer.train()
            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
            del trainer

        gc.collect()
        torch.cuda.empty_cache()

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training | stats: {stats}")

        # ================================================================
        # GET AIF RESULT (should be ready, runs <0.1s on CPU)
        # ================================================================
        aif_result = None
        if aif_future is not None:
            try:
                aif_result = aif_future.result(timeout=1.0)  # safe, should be instant
            except Exception as e:
                print(f"[Rank {rank}] AIF retrieval error: {e}")
        executor.shutdown(wait=False)

        # ================================================================
        # EARLY EXIT: IF AIF SOLVED PERFECTLY AND ONLY ONE TEST CASE
        # ================================================================
        test_cases = puzzle_ds.queries[key]['test']
        if len(test_cases) == 1 and aif_result and aif_result.get('is_solved', False) and aif_result.get('grid') is not None:
            solution_grid = np.array(aif_result['grid'])
            print(f"[Rank {rank}] ⭐ AIF PERFECTLY SOLVED {key}! Skipping 15-min inference.")
            dummy_result = [{
                "beam_score": 0.0,
                "score_aug": [0.0, 0.0],
                "solution": solution_grid
            }]
            # Correct base key: {key}_0 (matches split_multi_replies)
            with bz2.BZ2File(os.path.join(dir_outputs, f"{key}_0.out0"), "w") as f:
                pickle.dump(dummy_result, f)
            print(f"[Rank {rank}] finished {key} in {time.time() - start_time:.1f}s (AIF fast-path)")
            continue

        # ================================================================
        # FALLBACK: LLM INFERENCE WITH AIF STRUCTURAL PRUNING
        # ================================================================
        print(f"[Rank {rank}] AIF confidence {aif_result.get('confidence',0):.2f}. Running LLM inference...")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()
        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            for offset in [0, 4]:
                batches.append(subkeys[offset:offset+2])
            for offset in [2, 6]:
                batches.append(subkeys[offset:offset+2])
        for test_id, subkeys in test_id_to_subkeys.items():
            for offset in [8, 12]:
                batches.append(subkeys[offset:offset+2])
            for offset in [10, 14]:
                batches.append(subkeys[offset:offset+2])

        model = FastLanguageModel.for_inference(model)
        known_scores = {}
        expected_count = aif_result.get('expected_count', -1) if aif_result else -1

        with torch.inference_mode():
            for subkeys in batches:
                if time.time() - start_time > 1800 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {time.time()-start_time:.1f}s for {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")
                tokens = [tokenizer.encode(eval_ds.get(sk, formatter)["input"]) for sk in subkeys]
                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:
                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, token_seq in scored_beams:
                        array = formatter.convert_tokens_to_array(token_seq)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        # ============================================================
                        # AIF PRUNING: Penalize if generated object count mismatches
                        # ============================================================
                        if expected_count > 0:
                            bin_sol = (solution > 0).astype(np.int32)
                            _, gen_count = label(bin_sol)
                            if abs(gen_count - expected_count) > 0:
                                beam_score += abs(gen_count - expected_count) * 0.5

                        # ============================================================
                        # ORIGINAL SCORING LOGIC (unchanged)
                        # ============================================================
                        grid_id = (bk, tuple(map(tuple, solution)))
                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores

                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        print(f"[Rank {rank}] finished {key} in {time.time() - start_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

decoder.load_decoded_results("/kaggle/inference_outputs")

submission = data.get_submission(decoder.run_selection_algo())

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))